# 02. Partición Patient-Level

**Objetivo:** generar splits train/dev/test estrictos por paciente para evitar leakage.
**Entradas:** `data/ips_clean.csv`.
**Salidas:** `data/splits/dataset_base.csv`, `train_indices.csv`, `dev_indices.csv`, `test_indices.csv`.
**Notebook anterior:** `notebooks/pipeline/01_datos_eda_limpieza.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/03_denoising_reglas_core.ipynb`.


In [6]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

# Añadir la carpeta principal del proyecto al sistema para cargar funciones compartidas
sys.path.insert(0, str(Path.cwd().resolve()))
from utils_shared import setup_paths, validate_file_exists

paths = setup_paths()
DATA_PATH   = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
INPUT_FILE  = DATA_PATH / 'ips_clean.csv'

# Parámetros del experimento para reproducibilidad
RANDOM_STATE = 42
TEST_SIZE    = 0.20  # 20% de pacientes para evaluación final

print(f"Archivo de entrada: {INPUT_FILE.name}")


Archivo de entrada: ips_clean.csv


## 1. Cargar las notas clínicas limpias
Revisamos que el archivo del paso anterior (01_datos_eda_limpieza) exista y lo abrimos.


In [7]:
validate_file_exists(INPUT_FILE, 'Error: Falta el archivo limpio. Por favor ejecute pipeline/01_datos_eda_limpieza.ipynb primero.')

df = pd.read_csv(INPUT_FILE)
print(f"Lectura completada: {df.shape[0]} notas y {df.shape[1]} variables.")
display(df.head(3))


Lectura completada: 3143 notas y 4 variables.


,patient_id,etiqueta,texto,feat_had_template_block
0,406231,ansiedad,Reposicion de medicacion,1
1,406231,ansiedad,acude para reposicion,1
2,406231,ansiedad,"Se encuentra estable, tranquila, refiere buen ...",0


## 2. Definir el Diagnóstico Principal del Paciente
Dado que vamos a agrupar a nivel de personas (pacientes) y no de notas individuales, necesitamos saber si un paciente es de 'Ansiedad' o 'Depresión'. Si alguien tiene notas confusas, asignamos el diagnóstico que aparezca en la gran mayoría de sus notas.


In [8]:
PATIENT_COL = next((c for c in df.columns if c.lower() in ['patient_id', 'id_paciente', 'paciente', 'prontuario']), 'patient_id')
LABEL_COL   = 'etiqueta'

dropped = df[PATIENT_COL].isna().sum()
df = df.dropna(subset=[PATIENT_COL])
if dropped > 0:
    print(f"[Aviso] Se ignoraron {dropped} registros porque no tenían ID de paciente.")

try:
    df[PATIENT_COL] = df[PATIENT_COL].astype(int)
except:
    pass

def get_majority_label(patient_id, df):
    """
    Averigua cuál es el diagnóstico más repetido de un paciente específico.
    """
    labels = df[df[PATIENT_COL] == patient_id][LABEL_COL]
    return Counter(labels).most_common(1)[0][0]

patients = df[PATIENT_COL].unique()
patient_labels = {p: get_majority_label(p, df) for p in patients}

patients_df = pd.DataFrame({
    'patient_id': list(patient_labels.keys()),
    'label':      list(patient_labels.values())
})

print(f"Pacientes incluidos: {len(patients_df)}")
print("\nBalance general real a nivel de paciente:")
display(patients_df['label'].value_counts())


Pacientes incluidos: 90

Balance general real a nivel de paciente:


label
depresion    57
ansiedad     33
Name: count, dtype: int64

## 3. Realizar los 3 cortes (entrenamiento, desarrollo y prueba)
Separamos de forma proporcional (manteniendo la misma relación Ansiedad/Depresión). La fórmula es 60% entrenamiento, 20% desarrollo, 20% prueba.


In [9]:
# Primer corte: reservar el 20% para prueba final
patients_train_dev, patients_test = train_test_split(
    patients_df['patient_id'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=patients_df['label']
)

# Segundo corte: del 80% restante, usar 25% para desarrollo (equivale al 20% total).
patients_train, patients_dev = train_test_split(
    patients_train_dev,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=patients_df.set_index('patient_id').loc[patients_train_dev, 'label']
)

# Asignar cada nota según el grupo de su paciente
df['split'] = 'train'
df.loc[df[PATIENT_COL].isin(patients_dev), 'split'] = 'dev'
df.loc[df[PATIENT_COL].isin(patients_test), 'split'] = 'test'

print("Distribución final de notas clínicas:")
print(df['split'].value_counts())
print("\nProporción relativa del diagnóstico por grupo:")
display(df.groupby('split')[LABEL_COL].value_counts(normalize=True).unstack().round(3))


Distribución final de notas clínicas:
split
train    1911
test      637
dev       595
Name: count, dtype: int64

Proporción relativa del diagnóstico por grupo:


etiqueta,ansiedad,depresion
split,,
dev,0.313,0.687
test,0.253,0.747
train,0.302,0.698


## 4. Crear el Código de Barras (Tracking ID)
A cada nota le daremos un número seriado (`row_id`) que servirá como un GPS para rastrearla a lo largo de toda la investigación, incluso cuando cambie de algoritmos.


In [10]:
# Crear identificador único por nota
if 'row_id' not in df.columns:
    df = df.reset_index(drop=True)
    df.insert(0, 'row_id', df.index)

SPLITS_PATH.mkdir(parents=True, exist_ok=True)

for split_name in ['train', 'dev', 'test']:
    split_ids = df[df['split'] == split_name]['row_id']
    out_path = SPLITS_PATH / f'{split_name}_indices.csv'
    split_ids.to_csv(out_path, index=False, header=['row_id'])
    print(f"Índices guardados para '{split_name.upper()}': {len(split_ids)} notas.")

# Guardar conjunto de datos consolidado y ordenado
base_file = SPLITS_PATH / 'dataset_base.csv'
df.drop(columns=['split']).to_csv(base_file, index=False)
print(f"\nProceso completado. Dataset base guardado en: {base_file}")


Índices guardados para 'TRAIN': 1911 notas.
Índices guardados para 'DEV': 595 notas.
Índices guardados para 'TEST': 637 notas.

Proceso completado. Dataset base guardado en: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/splits/dataset_base.csv
